# 54. 조건부 열접촉 추가 오탐 확인 분기
53번 추가가스/정상 비교를 보고 만든 가설이므로 독립 검증으로 주장하지 않는다. 배럴5−금형 변화 불일치 또는 압력/충진시간의 조건부 편차가 낮다는 확인조건을 추가한다. 경계는 내부 정상5/10/20% 분위수. 전문가 경계는 기존52번97.5%를 고정한다. 외부정답으로 숫자를 맞추지 않는다.
각 고정 가설과 학습내부선택을 모두 보고한다. 기존47앵커는 유지하며 내부 선택은 원래RF기준의 추가효과 검사라 전체47앵커의 완전중첩 선택은 아니다. 센서 방향은 원인 정답이 아니다.


In [ ]:
from pathlib import Path
import json,time
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
anchor=pd.read_csv(tables/'rg3_innovation_seed_sensitivity_predictions.csv',parse_dates=['TimeStamp'])
anchor=anchor[anchor.model.eq('three_seed_mean')].set_index(['scheme','TimeStamp'])
oldinner=pd.read_csv(tables/'rg3_rank_fusion_inner_scores.csv',parse_dates=['TimeStamp'])
# Mechanism names are hypotheses, not gold-standard causal labels.
p=d.Max_Injection_Pressure
sw=d.Max_Switch_Over_Pressure
mold=(d.Mold_Temperature_3+d.Mold_Temperature_4)/2
barrel=d[[f'Barrel_Temperature_{i}' for i in range(1,7)]].mean(axis=1)
def ratio(a,b):return a/b.where(b.abs()>1e-6)
F={
 'pressure_support':pd.DataFrame({
 'peak_minus_switch':p-sw,
 'switch_fraction':ratio(sw,p),
 'cushion_per_meter_position':ratio(d.Cushion_Position,d.Plasticizing_Position),
 'pressure_dev10':d.Max_Injection_Pressure__dev10,
 'switch_pressure_delta':d.Max_Switch_Over_Pressure__delta1,
 'cushion_delta':d.Cushion_Position__delta1}),
 'surface_thermal_contact':pd.DataFrame({
 'barrel_mold_gap':barrel-mold,
 'mold_side_difference':d.Mold_Temperature_3-d.Mold_Temperature_4,
 'mold_delta_mean':(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2,
 'mold_dev_mean':(d.Mold_Temperature_3__dev10+d.Mold_Temperature_4__dev10)/2,
 'barrel_mold_delta_gap':d.Barrel_Temperature_5__delta1-(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2,
 'pressure_per_filling_time':ratio(p,d.Filling_Time)}),
 'material_release_proxy':pd.DataFrame({
 'barrel_spatial_spread':d[[f'Barrel_Temperature_{i}' for i in range(1,7)]].std(axis=1),
 'barrel6_minus5':d.Barrel_Temperature_6-d.Barrel_Temperature_5,
 'hopper_barrel_gap':barrel-d.Hopper_Temperature,
 'cycle_minus_plasticizing':d.Cycle_Time-d.Plasticizing_Time,
 'plasticizing_fraction':ratio(d.Plasticizing_Time,d.Cycle_Time),
 'back_pressure_dev10':d.Average_Back_Pressure__dev10})
}
F={'surface_thermal_contact':F['surface_thermal_contact']}
for name,x in F.items():
 x.replace([np.inf,-np.inf],np.nan,inplace=True)
 x.assign(TimeStamp=d.TimeStamp).to_csv(tables/f'rg3_pdf_{name}_features.csv',index=False,encoding='utf-8-sig')
 print('Feature group',name,list(x.columns),flush=True)
from sklearn.neighbors import NearestNeighbors
controls={
 'pressure_support':['Max_Injection_Speed','Filling_Time','Barrel_Temperature_5','Mold_Temperature_3'],
 'surface_thermal_contact':['Barrel_Temperature_5','Max_Injection_Speed','Filling_Time','Hopper_Temperature'],
 'material_release_proxy':['Plasticizing_Position','Max_Injection_Speed','Mold_Temperature_3','Average_Back_Pressure']
}
def local_residual(ref,q,name):
 normal=ref[y[ref]==0]
 raw=F[name].to_numpy(float)
 imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(raw[normal])
 raw=imp.transform(raw)
 context=d[controls[name]].to_numpy(float)
 ci=SimpleImputer(strategy='median',keep_empty_features=True).fit(context[normal])
 context=ci.transform(context)
 sc=RobustScaler().fit(context[normal]);context=sc.transform(context)
 k=min(31,len(normal))
 near=NearestNeighbors(n_neighbors=k).fit(context[normal])
 distances,neighbors=near.kneighbors(context[q])
 global_scale=np.std(raw[normal],axis=0)
 global_scale=np.maximum(global_scale*.1,1e-6)
 out=[]
 for pos,i in enumerate(q):
  indexes=normal[neighbors[pos]]
  indexes=indexes[indexes!=i][:30]
  if len(indexes)==0:indexes=normal
  center=np.median(raw[indexes],axis=0)
  scale=np.maximum(np.std(raw[indexes],axis=0),global_scale)
  out.append((raw[i]-center)/scale)
 return np.asarray(out)
memo={}
def predict(ref,q,name):
 key=(tuple(ref),tuple(q),name)
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 if len(np.unique(y[ref]))<2:return np.zeros((len(q),4))
 imp=SimpleImputer(strategy='median',keep_empty_features=True)
 tr=imp.fit_transform(local_residual(ref,ref,name));te=imp.transform(local_residual(ref,q,name))
 tree=[]
 for cls in [RandomForestClassifier,ExtraTreesClassifier]:
  seeds=[]
  for seed in [42,7,101]:
   model=cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2)
   seeds.append(model.fit(tr,y[ref]).predict_proba(te)[:,1])
  tree.append(np.mean(seeds,axis=0))
 # Fit scaling only on normals from the reference training data.
 sc=RobustScaler().fit(tr[y[ref]==0]);tx=sc.transform(tr);qx=sc.transform(te)
 linear=[]
 for C in [.1,1.,10.]:
  model=LogisticRegression(C=C,class_weight='balanced',max_iter=2000,random_state=42)
  linear.append(model.fit(tx,y[ref]).predict_proba(qx)[:,1])
 a,b=tree;c=np.mean(linear,axis=0)
 out=np.column_stack([(a+b)/2,(a+b+c)/3,np.sqrt(np.clip((a+b)/2*c,0,1)),c])
 memo[key]=out
 return out
models=['two_tree_mean','tree_linear_mean','tree_linear_geometric','logit3']


# Added gates are test-informed hypotheses; thresholds and inner selection use training only.
anchor=pd.read_csv(tables/'rg3_innovation_seed_sensitivity_predictions.csv',parse_dates=['TimeStamp'])
anchor=anchor[anchor.model.eq('three_seed_mean')].set_index(['scheme','TimeStamp'])
oldinner=pd.read_csv(tables/'rg3_rank_fusion_inner_scores.csv',parse_dates=['TimeStamp'])
records=[];choices=[];candidates=[];foldaudit=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
 for j,b in enumerate(blocks):
  q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
  if len(np.unique(y[ref]))<2:continue
  outer=predict(ref,q,'surface_thermal_contact')[:,1]
  zo=local_residual(ref,q,'surface_thermal_contact')
  inner=np.full(len(ref),np.nan);zi=np.full((len(ref),6),np.nan)
  for ib in groups.iloc[ref].unique():
   mask=groups.iloc[ref].eq(ib).to_numpy()
   inner[mask]=predict(ref[~mask],ref[mask],'surface_thermal_contact')[:,1]
   zi[mask]=local_residual(ref[~mask],ref[mask],'surface_thermal_contact')
  anc=anchor.loc[(scheme,d.TimeStamp.iloc[q]),'prediction'].to_numpy(bool)
  ic=oldinner[(oldinner.scheme==scheme)&(oldinner.heldout_block==b)].set_index('TimeStamp').loc[d.TimeStamp.iloc[ref]]
  cb=float(np.quantile(ic.loc[y[ref]==0,'raw_baseline_score'],.99,method='higher'))
  oldpr=ic.raw_baseline_score.gt(cb).to_numpy()
  tn0,fp0,fn0,tp0=confusion_matrix(y[ref],oldpr,labels=[0,1]).ravel()
  f0=2*tp0/(2*tp0+fp0+fn0) if 2*tp0+fp0+fn0 else 0
  ns=(y[ref]==0)&np.isfinite(inner)
  cut=float(np.quantile(inner[ns],.975,method='higher'))
  local=[]
  for feature,columns in [('thermal_change_low',[4]),('pressure_support_low',[5]),('joint_thermal_pressure_low',[4,5])]:
   for quant in [.05,.1,.2]:
    thresholds=np.quantile(zi[ns][:,columns],quant,axis=0)
    go=(zo[:,columns]<thresholds).all(axis=1)
    gi=(zi[:,columns]<thresholds).all(axis=1)
    pr=oldpr|((inner>cut)&gi)
    tn,fp,fn,tp=confusion_matrix(y[ref],pr,labels=[0,1]).ravel()
    fi=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
    allow=bool(tp>tp0 and fp-fp0<=2 and fi>f0)
    name=feature+'_'+str(quant)
    row=dict(scheme=scheme,block=b,gate=name,inner_added_TP=int(tp-tp0),inner_added_FP=int(fp-fp0),inner_F1=fi,allowed=allow,score_threshold=cut,gate_thresholds=json.dumps(thresholds.tolist()))
    candidates.append(row)
    extra=(outer>cut)&go
    local.append((row,extra))
    for k,i in enumerate(q):records.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),prediction=int(anc[k]|extra[k]),anchor_prediction=int(anc[k]),score=float(outer[k]),score_threshold=cut,gate_value_thermal=float(zo[k,4]),gate_value_pressure=float(zo[k,5]),gate_pass=int(go[k])))
  valid=[c for c in local if c[0]['allowed']]
  best=max(valid,key=lambda c:(c[0]['inner_F1'],-c[0]['inner_added_FP'])) if valid else None
  pr=anc if best is None else anc|best[1]
  choices.append(dict(scheme=scheme,block=b,selected='none' if best is None else best[0]['gate'],inner_added_TP=0 if best is None else best[0]['inner_added_TP'],inner_added_FP=0 if best is None else best[0]['inner_added_FP']))
  for k,i in enumerate(q):records.append(dict(scheme=scheme,model='inner_guarded_selector',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),prediction=int(pr[k]),anchor_prediction=int(anc[k]),score=float(outer[k]),score_threshold=cut,gate_value_thermal=float(zo[k,4]),gate_value_pressure=float(zo[k,5]),gate_pass=np.nan))
  foldaudit.append(dict(scheme=scheme,block=b,train=len(ref),test=len(q),train_gas=int(y[ref].sum()),test_gas=int(y[q].sum()),timestamp_overlap=len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q]))))
  print('Completed confirm gates',scheme,b,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,model),g in pp.groupby(['scheme','model'],sort=False):
 assert not g.TimeStamp.duplicated().any()
 assert ((g.anchor_prediction==0)|(g.prediction==1)).all()
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(scheme=scheme,model=model,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
mm=pd.DataFrame(rows)
for obj,name in [(pp,'predictions'),(mm,'metrics'),(pd.DataFrame(choices),'choices'),(pd.DataFrame(candidates),'inner_candidates'),(pd.DataFrame(foldaudit),'fold_audit')]:
 obj.to_csv(tables/f'rg3_pdf_confirm_gates_{name}.csv',index=False,encoding='utf-8-sig')
print(mm.to_string(index=False))
